### Exercise 1

Take a DNA sequence from NCBI (an entire genome) and take 2000 random samples of about 100-120 bases from it; store these samples in a list.
Rebuild the DNA sequence using these random samples and find out what the main problem will be without the algorithmic approach.

Genome used: *Enterobacteria phage lambda*, complete genome (NCBI **NC_001416.1**, 48,502 bp).

In [3]:
import random

def take_samples(genom, n=2000, min_len=100, max_len=120, seed=42):
    rng = random.Random(seed)
    samples, starts = [],[]
    for _ in range(n):
        length = rng.randint(min_len,max_len)
        start = rng.randint(0, len(genom) - length)
        samples.append(genom[start:start + length])
        starts.append(start)
    return samples,starts

with open("lambda_phage.fasta") as f:
    genome = "".join(line.strip() for line in f if not line.startswith(">")).upper()

samples, starts = take_samples(genome)
total_bases = sum(len(s) for s in samples)

print("Genome: Enterobacteria phage lambda (NC_001416.1)")
print(f"Genome length   = {len(genome):,} bases")
print(f"Samples         = {len(samples):,} (lengths {min(map(len, samples))}-{max(map(len, samples))})")
print(f"Bases in samples = {total_bases:,}  ->  average coverage = {total_bases / len(genome):.2f}x")
print()
for i in range(3):
    print(f"sample {i+1}: {samples[i][:60]}...")

Genome: Enterobacteria phage lambda (NC_001416.1)
Genome length   = 48,502 bases
Samples         = 2,000 (lengths 100-120)
Bases in samples = 220,554  ->  average coverage = 4.55x

sample 1: CCGTGTGGCAGAGCTGAAAGAGGAGCTTGATGACACGGATGAAACTGCCGGTCAGGACAC...
sample 2: AACGCCAGCAGACTGGAGGAGTTTTCGAAAGAGTGGAAGGATGCCAGTGATAAGTGGAAT...
sample 3: GTTGTCAGCGGCACAGCCGGGCGATGTGCTGCTGTGCTGTTTTGGTTCATCAGTGCCGAA...


#### Attempt 1: rebuild without any algorithm

The simplest idea is to put the samples back together one after another, in the order we have them.

In [4]:
rebuilt_dna = "".join(samples)
print(f"Rebuilt length  = {len(rebuilt_dna):,} bases (original: {len(genome):,})")
print(f"Identical to the genome: {rebuilt_dna == genome}")

Rebuilt length  = 220,554 bases (original: 48,502)
Identical to the genome: False


The result is about 4.5 times longer than the genome and is nonsense: the samples are random, so we do not know
where each one came from, they overlap each other, and many regions appear several times.
To rebuild the genome we have to use the only information the samples carry: **overlaps** between them.

#### Attempt 2: greedy overlap merging

1. remove duplicate samples and samples contained inside another sample;
2. for every pair (A, B) find the longest suffix of A that equals a prefix of B (at least 20 bases);
3. repeatedly join the pair with the largest overlap, until no join is possible.

In [ ]:
from collections import defaultdict

# 1. Remove duplicate samples and samples contained inside another sample
unique_samples = sorted(list(set(samples)), key=len, reverse=True)
filtered_samples = []
for s in unique_samples:
    if not any(s in other for other in filtered_samples):
        filtered_samples.append(s)

print(f"Initial samples: {len(samples)} -> Filtered unique samples: {len(filtered_samples)}")

# 2. Find longest suffix of A that equals a prefix of B (at least 20 bases)
min_overlap = 20
kmer_prefix = defaultdict(list)
for j, read in enumerate(filtered_samples):
    kmer_prefix[read[:min_overlap]].append(j)

overlaps = []
for i, a in enumerate(filtered_samples):
    for pos in range(len(a) - min_overlap, -1, -1):
        kmer = a[pos:pos + min_overlap]
        if kmer in kmer_prefix:
            for j in kmer_prefix[kmer]:
                if i != j and filtered_samples[j].startswith(a[pos:]):
                    overlaps.append((len(a) - pos, i, j))

# 3. Repeatedly join the pair with the largest overlap (Greedy assembly)
overlaps.sort(key=lambda x: x[0], reverse=True)

succ = {}
pred = {}
ov_len = {}
parent = {i: i for i in range(len(filtered_samples))}

def find_root(i):
    if parent[i] == i:
        return i
    parent[i] = find_root(parent[i])
    return parent[i]

for ov, i, j in overlaps:
    if i not in succ and j not in pred:
        root_i, root_j = find_root(i), find_root(j)
        if root_i != root_j:  # prevent forming circular loops
            parent[root_i] = root_j
            succ[i] = j
            pred[j] = i
            ov_len[i] = ov

# Rebuild the resulting contigs
start_nodes = [i for i in range(len(filtered_samples)) if i not in pred]
contigs = []
for start in start_nodes:
    curr = start
    seq = filtered_samples[curr]
    while curr in succ:
        nxt = succ[curr]
        seq += filtered_samples[nxt][ov_len[curr]:]
        curr = nxt
    contigs.append(seq)

contigs.sort(key=len, reverse=True)
print(f"Reconstructed contigs: {len(contigs)}")
print(f"Longest contig: {len(contigs[0]):,} bp, Shortest contig: {len(contigs[-1]):,} bp")
print(f"Total length of contigs: {sum(len(c) for c in contigs):,} bp (genome: {len(genome):,} bp)")
print(f"Top 5 contig lengths: {[len(c) for c in contigs[:5]]}")


The genome is **not** rebuilt as a single sequence - we get many separate pieces (contigs). Let's see why, by looking at
how many samples cover each position of the genome (here we use the real start positions, which the rebuild never had).

In [ ]:
import matplotlib.pyplot as plt

# Calculate per-base coverage across the genome
coverage = [0] * len(genome)
for s, sample in zip(starts, samples):
    for pos in range(s, s + len(sample)):
        coverage[pos] += 1

zero_coverage_bases = sum(1 for c in coverage if c == 0)
print(f"Positions with zero coverage (gaps): {zero_coverage_bases:,} bases ({zero_coverage_bases / len(genome) * 100:.2f}% of genome)")
print(f"Mean coverage: {sum(coverage) / len(genome):.2f}x (Max: {max(coverage)}x, Min: {min(coverage)}x)")

# Plot coverage across the genome
plt.figure(figsize=(14, 4))
plt.plot(coverage, color="steelblue", linewidth=0.8, label="Read Coverage")
plt.fill_between(range(len(genome)), coverage, color="skyblue", alpha=0.4)
plt.axhline(0, color="red", linestyle="--", linewidth=1, label="Zero coverage (gaps)")
plt.title("Sequencing Coverage across Enterobacteria phage lambda Genome")
plt.xlabel("Genome Position (bp)")
plt.ylabel("Coverage (number of reads)")
plt.legend()
plt.tight_layout()
plt.show()


#### The repeat problem

Phage lambda has almost no long repeats, so all contigs above are correct. Real genomes (bacteria, humans) have many.
If a region longer than a sample appears twice, the samples from both copies look identical and the merge cannot know
which way to continue. A small example: `UNIQUE_A + REPEAT + UNIQUE_B + REPEAT + UNIQUE_C`.

In [1]:
# Small demonstration of the repeat problem
# Genome structure: UNIQUE_A + REPEAT + UNIQUE_B + REPEAT + UNIQUE_C
A = "AAAGGGTTT"
B = "CCCTTTGGG"
C = "TTTAAACCC"
REPEAT = "TATATATATATA"

synthetic_genome = A + REPEAT + B + REPEAT + C
print("Original synthetic genome:")
print(synthetic_genome)
print(f"Length: {len(synthetic_genome)} bp\n")

# Reads shorter than repeat (samples cannot span across the entire repeat)
toy_reads = [
    A + REPEAT[:4],             # crosses A into REPEAT
    REPEAT[2:10],               # entirely inside REPEAT (ambiguous)
    REPEAT[6:] + B[:3],         # crosses REPEAT into B
    B + REPEAT[:3],             # crosses B into REPEAT
    REPEAT[5:] + C[:4],         # crosses REPEAT into C
    C                           # inside C
]

print("Toy reads sampled:")
for r in toy_reads:
    print(" ", r)

print("\nWhy merging fails:")
print("1. The repeat is longer than read overlaps, so the assembler cannot tell")
print("   whether the first REPEAT continues to B or skips directly to C.")
print("2. Without long reads or paired-end reads, the assembler makes a wrong join")
print("   (e.g., A + REPEAT + C), misassembling or deleting entire section B!")


Original synthetic genome:
AAAGGGTTTTATATATATATACCCTTTGGGTATATATATATATTTAAACCC
Length: 51 bp

Toy reads sampled:
  AAAGGGTTTTATA
  TATATATA
  TATATACCC
  CCCTTTGGGTAT
  ATATATATTTA
  TTTAAACCC

Why merging fails:
1. The repeat is longer than read overlaps, so the assembler cannot tell
   whether the first REPEAT continues to B or skips directly to C.
2. Without long reads or paired-end reads, the assembler makes a wrong join
   (e.g., A + REPEAT + C), misassembling or deleting entire section B!


### Conclusion - What is the main problem?

Without an algorithmic approach, a genome cannot be rebuilt because:

1. **No Coordinates (Scrambled Order):**
   Reads have no labels indicating their genomic origin. Naive concatenation yields a sequence **~4.5x too long** and completely disordered.

2. **Repeats (The Biggest Challenge):**
   Identical repeat sequences create ambiguous overlaps. Short reads cannot span across repeats, causing misassemblies, loops, or dropped sections.

3. **Coverage Gaps:**
   Random sampling leaves uncovered gaps by chance, breaking the assembly into isolated pieces (**contigs**).

4. **Computational Complexity :**
   ($O(n^2)$)